# Overlap analysis demo (inline)

This notebook reproduces the overlap and Venn steps inline (no external script).


In [ ]:
# Inputs
from pathlib import Path
import pandas as pd

results_dir = Path("results/Supp_Data_4_hg38_mouse_mm10_converted_w_human_sc")
excel_path = results_dir / "Supp_Data_4_hg38_mouse_mm10_converted_w_human_sc.xlsx"
sheet_a = "Human_hg38_scATAC_combined"
sheet_b = "Mouse_Converted_mm10_to_hg38"
coding_bed = results_dir / "hg38_coding_regions_refseq.bed"

a_df = pd.read_excel(excel_path, sheet_name=sheet_a)
b_df = pd.read_excel(excel_path, sheet_name=sheet_b)
# Normalize columns
lower_a = {c.lower(): c for c in a_df.columns}
lower_b = {c.lower(): c for c in b_df.columns}
a = a_df[[lower_a['chr'], lower_a['start'], lower_a['end']]].rename(columns={lower_a['chr']:'chr', lower_a['start']:'start', lower_a['end']:'end'})
b = b_df[[lower_b['chr'], lower_b['start'], lower_b['end']]].rename(columns={lower_b['chr']:'chr', lower_b['start']:'start', lower_b['end']:'end'})

len(a), len(b)


In [ ]:
# Use bedtools to sort/merge, subtract, and intersect
import subprocess, tempfile

def write_bed(df: pd.DataFrame, path: Path) -> Path:
    df.to_csv(path, sep='\t', header=False, index=False)
    return path

work = Path(tempfile.mkdtemp(prefix="overlap_nb_"))
A_raw, B_raw = work/"A.raw.bed", work/"B.raw.bed"
write_bed(a, A_raw)
write_bed(b, B_raw)

def sort_merge(src: Path, dst: Path) -> None:
    p1 = subprocess.Popen(["sort", "-k1,1", "-k2,2n", str(src)], stdout=subprocess.PIPE, text=True)
    p2 = subprocess.Popen(["bedtools", "merge", "-i", "-"], stdin=p1.stdout, stdout=subprocess.PIPE, text=True)
    p1.stdout.close()  # type: ignore
    out, _ = p2.communicate()
    dst.write_text(out or "")

A_m, B_m = work/"A.merged.bed", work/"B.merged.bed"
sort_merge(A_raw, A_m)
sort_merge(B_raw, B_m)

# Unique sets
A_only, B_only = work/"A.only.bed", work/"B.only.bed"
A_only.write_text(subprocess.run(["bedtools", "subtract", "-a", str(A_m), "-b", str(B_m)], capture_output=True, text=True).stdout or "")
B_only.write_text(subprocess.run(["bedtools", "subtract", "-a", str(B_m), "-b", str(A_m)], capture_output=True, text=True).stdout or "")

# Mutual
mut_out = work/"mutual.bed"
proc = subprocess.run(["bedtools", "intersect", "-a", str(A_m), "-b", str(B_m), "-wo"], capture_output=True, text=True)
lines = []
mut_bp = 0
for ln in (proc.stdout or "").splitlines():
    p = ln.split("\t")
    if len(p) < 7:
        continue
    chr_a, sa, ea = p[0], int(p[1]), int(p[2])
    chr_b, sb, eb = p[3], int(p[4]), int(p[5])
    ov = int(p[-1])
    mut_bp += max(ov, 0)
    if chr_a != chr_b:
        continue
    s, e = max(sa, sb), min(ea, eb)
    if s < e:
        lines.append(f"{chr_a}\t{s}\t{e}\n")
(work/"mut.tmp.bed").write_text("".join(lines))
sort_merge(work/"mut.tmp.bed", mut_out)

# Totals
awk = lambda p: int((subprocess.run(["awk", "-F\t", "{s+=($3-$2)} END {print s}", str(p)], capture_output=True, text=True).stdout or "0").strip() or "0")
Aonly_bp = awk(A_only)
Bonly_bp = awk(B_only)
mutual_bp = mut_bp
Atotal_bp = Aonly_bp + mutual_bp
Btotal_bp = Bonly_bp + mutual_bp

Aonly_bp, Bonly_bp, mutual_bp, Atotal_bp, Btotal_bp


In [ ]:
# Compute coding/non-coding splits for each group

def coding_bp(bed_path: Path, coding_bed: Path) -> int:
    proc = subprocess.run(["bedtools", "intersect", "-a", str(bed_path), "-b", str(coding_bed), "-wo"], capture_output=True, text=True)
    total = 0
    for ln in (proc.stdout or "").splitlines():
        try:
            total += int(ln.split("\t")[-1])
        except Exception:
            pass
    return total

coding_mut = coding_bp(work/"mutual.bed", coding_bed)
coding_A = coding_bp(work/"A.only.bed", coding_bed)
coding_B = coding_bp(work/"B.only.bed", coding_bed)

summary = pd.DataFrame([
    {"group":"mutual", "total_bp": mutual_bp, "coding_bp": coding_mut},
    {"group":"unique_A", "total_bp": Aonly_bp, "coding_bp": coding_A},
    {"group":"unique_B", "total_bp": Bonly_bp, "coding_bp": coding_B},
])
summary["noncoding_bp"] = (summary["total_bp"] - summary["coding_bp"]).clip(lower=0)
summary["coding_pct"] = (summary["coding_bp"] / summary["total_bp"] * 100.0).fillna(0.0)
summary["noncoding_pct"] = (summary["noncoding_bp"] / summary["total_bp"] * 100.0).fillna(0.0)
summary


In [ ]:
# Load genome total bp for percentages under labels
import json
resources_json = Path("bed_file_merger/resources/genome_info.json")
with resources_json.open("r") as f:
    gi = json.load(f)
GB = "hg38"
GENOME_TOTAL = int(gi.get(GB, {}).get("Total bases", 0))
GENOME_TOTAL


In [ ]:
# Draw Venns (no-text and with-text) similarly to the script/notebook
import matplotlib.pyplot as plt
from matplotlib_venn import venn2
import seaborn as sns

fig_dir = results_dir / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

base_a = sheet_a.replace(" ", "_")
base_b = sheet_b.replace(" ", "_")

# No text
fig = plt.figure(figsize=(6, 4), dpi=300)
v = venn2(subsets=(Aonly_bp, Bonly_bp, mutual_bp), set_labels=(sheet_a, sheet_b))
pal = sns.color_palette("pastel", 12)
if v.get_patch_by_id('10'): v.get_patch_by_id('10').set_color(pal[-3]); v.get_patch_by_id('10').set_alpha(0.7)
if v.get_patch_by_id('01'): v.get_patch_by_id('01').set_color(pal[-7]); v.get_patch_by_id('01').set_alpha(0.7)
if v.get_patch_by_id('11'): v.get_patch_by_id('11').set_color(pal[-2]); v.get_patch_by_id('11').set_alpha(0.7)
for lid in ['10','01','11']:
    if v.get_label_by_id(lid): v.get_label_by_id(lid).set_text("")
for lbl in (v.set_labels or []):
    if lbl: lbl.set_text("")
plt.tight_layout()
(fig_dir / f"venn_no_text_{base_a}_vs_{base_b}.svg").write_bytes(fig.canvas.tostring_svg())
fig.savefig(fig_dir / f"venn_no_text_{base_a}_vs_{base_b}.png", dpi=300, transparent=True)
plt.close(fig)

# With text
fig = plt.figure(figsize=(10, 6), dpi=300)
v = venn2(subsets=(Aonly_bp, Bonly_bp, mutual_bp), set_labels=(sheet_a, sheet_b))
if v.get_patch_by_id('10'): v.get_patch_by_id('10').set_color(pal[-3]); v.get_patch_by_id('10').set_alpha(0.7)
if v.get_patch_by_id('01'): v.get_patch_by_id('01').set_color(pal[-7]); v.get_patch_by_id('01').set_alpha(0.7)
if v.get_patch_by_id('11'): v.get_patch_by_id('11').set_color(pal[-2]); v.get_patch_by_id('11').set_alpha(0.7)
label_fontsize = 15
venn_label_fontsize = 13
if v.set_labels:
    for lbl in v.set_labels:
        if lbl: lbl.set_fontsize(label_fontsize)
# percents
A_unique_pct = (Aonly_bp/Atotal_bp*100.0) if Atotal_bp>0 else 0.0
B_unique_pct = (Bonly_bp/Btotal_bp*100.0) if Btotal_bp>0 else 0.0
A_shared_pct = (mutual_bp/Atotal_bp*100.0) if Atotal_bp>0 else 0.0
B_shared_pct = (mutual_bp/Btotal_bp*100.0) if Btotal_bp>0 else 0.0
if v.get_label_by_id('10'): v.get_label_by_id('10').set_text(f"{Aonly_bp:,}bp\n({A_unique_pct:.1f}%)"); v.get_label_by_id('10').set_fontsize(venn_label_fontsize)
if v.get_label_by_id('01'): v.get_label_by_id('01').set_text(f"{Bonly_bp:,}bp\n({B_unique_pct:.1f}%)"); v.get_label_by_id('01').set_fontsize(venn_label_fontsize)
if v.get_label_by_id('11'): v.get_label_by_id('11').set_text(f"{mutual_bp:,}bp\n({A_shared_pct:.1f}% of {sheet_a})\n({B_shared_pct:.1f}% of {sheet_b})"); v.get_label_by_id('11').set_fontsize(venn_label_fontsize)
# totals + genome
A_genome_pct = (Atotal_bp/GENOME_TOTAL*100.0) if GENOME_TOTAL>0 else 0.0
B_genome_pct = (Btotal_bp/GENOME_TOTAL*100.0) if GENOME_TOTAL>0 else 0.0
if v.set_labels:
    la = v.set_labels[0]
    if la:
        xa, ya = la.get_position()
        plt.text(xa-0.04, ya-0.01, f"\n{Atotal_bp:,}bp\n({A_genome_pct:.2f}% of {GB} genome)", ha='center', va='top', fontsize=venn_label_fontsize)
    lb = v.set_labels[1]
    if lb and v.set_labels[0]:
        xb, yb = lb.get_position()
        xa0, ya0 = v.set_labels[0].get_position()
        v.set_labels[1].set_position((xb, ya0))
        xb2, y2 = v.set_labels[1].get_position()
        plt.text(xb2+0.09, y2-0.01, f"\n{Btotal_bp:,}bp\n({B_genome_pct:.2f}% of {GB} genome)", ha='center', va='top', fontsize=venn_label_fontsize)
plt.tight_layout()
(fig_dir / f"venn_with_text_{base_a}_vs_{base_b}.svg").write_bytes(fig.canvas.tostring_svg())
fig.savefig(fig_dir / f"venn_with_text_{base_a}_vs_{base_b}.png", dpi=300, transparent=True)
plt.close(fig)

print("Saved to:", fig_dir)
